In [1]:
import numpy as np
import pandas as pd
import statistics

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_validate, train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)


In [2]:
data = pd.read_csv("../data/data2.csv", index_col=0)
print(data)

      country time_of_day        lat       long       road_type  \
8459       SE         day  55.604209  13.028574            city   
63960      DE         day  50.936889   6.908079  arterial-urban   
71801      IT         day  41.987014  12.496538         highway   
90649      DE         day  49.182153   9.414737         highway   
48806      SE         day  55.576732  13.013051            city   
...       ...         ...        ...        ...             ...   
11242      SE         day  55.598127  12.975875            city   
90673      PL         day  50.253217  19.823732   smaller-rural   
24373      DE         day  50.931399   6.953686            city   
41597      SE         day  55.608149  13.003458            city   
40353      IT       night  41.918667  12.383545            city   

      road_condition            weather  solar_angle_elevation  month  hour  \
8459          normal             cloudy              36.560248      5    14   
63960         normal               ra

In [3]:
data = data.dropna().reset_index(drop=True)
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 7,985 rows and 38 columns


In [4]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["conf"]

data = data.drop(columns=["iou", "lrp"])
data_indices = data.index.to_numpy()

In [5]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 7985
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast',
       'country', 'distortion_magnitude', 'field_view_horizontal',
       'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat',
       'lateral_acceleration', 'lateral_velocity', 'long', 'month',
       'noisiness', 'quality', 'rain', 'relative_humidity_2m',
       'road_condition', 'road_type', 'sharpness', 'snowfall',
       'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather',
       'wind_speed_10m'],
      dtype='object')
Number of columns: 33


In [6]:
numeric_columns = ['conf', 'solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']

categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'month', 'noisiness', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


# Assessor Tests

Split data into train 60% and validation 40%

In [7]:
#X_train, X_test, y_train, y_test = train_test_split(data, test_size=0.4, train_size=0.6, stratify=data["iou"])
train_idx, test_idx = train_test_split(data_indices, test_size=0.4, train_size=0.6)

X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print(X_train.columns)

X: 4791 3194
y: 4791 3194
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour',
       'forward_acceleration', 'lateral_acceleration', 'forward_velocity',
       'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m',
       'weather_code', 'conf'],
      dtype='object')


In [8]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [9]:
mean_iou_train = np.mean(y_train)
iou_pred_test = np.full_like(y_test, mean_iou_train)
mean_baseline_r2 = r2_score(y_test, iou_pred_test)
mean_baseline_mae = np.mean(np.abs(y_test - iou_pred_test))
mean_baseline_mse = np.mean((y_test - iou_pred_test)**2)
print(f"R2 score of mean baseline: {mean_baseline_r2:.4f}")
print(f"MAE of mean baseline: {mean_baseline_mae:.4f}")
print(f"MSE of mean baseline: {mean_baseline_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Constant Mean Predictor',
    'test_r2': mean_baseline_r2,
    'test_mae': mean_baseline_mae,
    'test_mse': mean_baseline_mse,
})


R2 score of mean baseline: -0.0001
MAE of mean baseline: 0.1131
MSE of mean baseline: 0.0232


#### Linear Reg on Conf

In [10]:

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
cv_results_conf = cross_validate(
    linear_reg_conf,
    conf_train,
    y_train,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results_conf['train_r2'])
mean_test_r2 = statistics.mean(cv_results_conf['test_r2'])
mean_train_mae = -statistics.mean(cv_results_conf['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results_conf['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results_conf['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results_conf['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.2992
Mean CV test r2 score 0.2953
Mean CV train MAE score 0.0950
Mean CV test MAE score 0.0951
Mean CV train MSE score 0.0157
Mean CV test MSE score 0.0157


In [11]:
linear_reg_conf.fit(conf_train, y_train)
y_pred = linear_reg_conf.predict(conf_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Univariate Linear Regression (Confidence)',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3416
MAE test score 0.0936
MSE test score 0.0153


#### Linear Regression

Train models with cv and then test.

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
cv_results = cross_validate(
    linear_reg,
    X_train,
    y_train,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.3382
Mean CV test r2 score 0.3113
Mean CV train MAE score 0.0924
Mean CV test MAE score 0.0938
Mean CV train MSE score 0.0148
Mean CV test MSE score 0.0154


In [13]:
linear_reg.fit(X_train, y_train)
y_pred = linear_reg.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Linear Regression',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3519
MAE test score 0.0926
MSE test score 0.0150


#### Decision Trees

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor())
])
cv_results = cross_validate(
    decision_tree,
    X_train,
    y_train,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 1.0000
Mean CV test r2 score -0.4110
Mean CV train MAE score -0.0000
Mean CV test MAE score 0.1309
Mean CV train MSE score -0.0000
Mean CV test MSE score 0.0315


In [15]:
decision_tree.fit(X_train, y_train)
y_pred = decision_tree.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Decision Tree',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score -0.3400
MAE test score 0.1305
MSE test score 0.0310


#### Random Forest

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor())
])
cv_results = cross_validate(
    rand_forest,
    X_train,
    y_train,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.9038
Mean CV test r2 score 0.3230
Mean CV train MAE score 0.0343
Mean CV test MAE score 0.0917
Mean CV train MSE score 0.0022
Mean CV test MSE score 0.0151


In [17]:
rand_forest.fit(X_train, y_train)
y_pred = rand_forest.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Random Forest',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3423
MAE test score 0.0920
MSE test score 0.0152


#### MLP

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor())
])
cv_results = cross_validate(
    mlp,
    X_train,
    y_train,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.2405
Mean CV test r2 score -0.0714
Mean CV train MAE score 0.1004
Mean CV test MAE score 0.1160
Mean CV train MSE score 0.0170
Mean CV test MSE score 0.0240


In [19]:
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'MLP',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.0385
MAE test score 0.1135
MSE test score 0.0223


### LRP


#### Baseline

Predict Only the Mean


In [20]:
mean_lrp_train = np.mean(y_train_lrp)
lrp_pred_test = np.full_like(y_test_lrp, mean_lrp_train)
mean_lrp_r2 = r2_score(y_test_lrp, lrp_pred_test)
mean_lrp_mae = np.mean(np.abs(y_test_lrp - lrp_pred_test))
mean_lrp_mse = np.mean((y_test_lrp - lrp_pred_test)**2)
print(f"R2 score of random baseline: {mean_lrp_r2:.4f}")
print(f"MAE of random baseline: {mean_lrp_mae:.4f}")
print(f"MSE of random baseline: {mean_lrp_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Constant Mean Predictor',
    'test_r2': mean_lrp_r2,
    'test_mae': mean_lrp_mae,
    'test_mse': mean_lrp_mse,
})


R2 score of random baseline: -0.0001
MAE of random baseline: 0.0964
MSE of random baseline: 0.0175


#### Linear Reg on Conf

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_lrp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
cv_results_conf_lrp = cross_validate(
    linear_reg_conf_lrp,
    conf_train,
    y_train_lrp,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results_conf_lrp['train_r2'])
mean_test_r2 = statistics.mean(cv_results_conf_lrp['test_r2'])
mean_train_mae = -statistics.mean(cv_results_conf_lrp['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results_conf_lrp['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results_conf_lrp['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results_conf_lrp['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.3784
Mean CV test r2 score 0.3745
Mean CV train MAE score 0.0760
Mean CV test MAE score 0.0760
Mean CV train MSE score 0.0106
Mean CV test MSE score 0.0106


In [22]:
linear_reg_conf.fit(conf_train, y_train_lrp)
y_pred = linear_reg_conf.predict(conf_test)
iou_test_r2 = r2_score(y_test_lrp, y_pred)
iou_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
iou_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Univariate Linear Regression (Confidence)',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.4008
MAE test score 0.0750
MSE test score 0.0105


#### Linear Regression


Train models with cv and then test.


In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
cv_results = cross_validate(
    linear_reg_lrp,
    X_train,
    y_train_lrp,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.4199
Mean CV test r2 score 0.3970
Mean CV train MAE score 0.0731
Mean CV test MAE score 0.0741
Mean CV train MSE score 0.0098
Mean CV test MSE score 0.0102


In [24]:
linear_reg_lrp.fit(X_train, y_train_lrp)
y_pred = linear_reg_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Linear Regression',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4205
MAE test score 0.0735
MSE test score 0.0101


#### Decision Trees


In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor())
])
cv_results = cross_validate(
    decision_tree_lrp,
    X_train,
    y_train_lrp,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 1.0000
Mean CV test r2 score -0.2382
Mean CV train MAE score -0.0000
Mean CV test MAE score 0.1035
Mean CV train MSE score -0.0000
Mean CV test MSE score 0.0208


In [26]:
decision_tree_lrp.fit(X_train, y_train_lrp)
y_pred = decision_tree_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Decision Tree',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score -0.0781
MAE test score 0.1005
MSE test score 0.0188


#### Random Forest


In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor())
])
cv_results = cross_validate(
    rand_forest_lrp,
    X_train,
    y_train_lrp,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.9146
Mean CV test r2 score 0.3782
Mean CV train MAE score 0.0272
Mean CV test MAE score 0.0737
Mean CV train MSE score 0.0014
Mean CV test MSE score 0.0105


In [28]:
rand_forest_lrp.fit(X_train, y_train_lrp)
y_pred = rand_forest_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Random Forest',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4242
MAE test score 0.0722
MSE test score 0.0100


#### MLP


In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor())
])
cv_results = cross_validate(
    mlp_lrp,
    X_train,
    y_train_lrp,
    cv=10,
    scoring=('r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'),
    return_train_score=True
)
mean_train_r2 = statistics.mean(cv_results['train_r2'])
mean_test_r2 = statistics.mean(cv_results['test_r2'])
mean_train_mae = -statistics.mean(cv_results['train_neg_mean_absolute_error'])
mean_test_mae = -statistics.mean(cv_results['test_neg_mean_absolute_error'])
mean_train_mse = -statistics.mean(cv_results['train_neg_mean_squared_error'])
mean_test_mse = -statistics.mean(cv_results['test_neg_mean_squared_error'])
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")


Mean CV train r2 score 0.2576
Mean CV test r2 score -0.1075
Mean CV train MAE score 0.0851
Mean CV test MAE score 0.0995
Mean CV train MSE score 0.0126
Mean CV test MSE score 0.0186


In [30]:
mlp_lrp.fit(X_train, y_train_lrp)
y_pred = mlp_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'MLP',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score -0.2123
MAE test score 0.0989
MSE test score 0.0212


### Final Model Comparison


In [31]:
results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(2)
results_df["test_mse"] = results_df["test_mse"].round(2)
results_df.to_csv("results.csv")
display(results_df)


,target,model,test_r2,test_mae,test_mse
0,IoU,Constant Mean Predictor,-0.0001,0.11,0.02
1,IoU,Univariate Linear Regression (Confidence),0.3416,0.09,0.02
2,IoU,Linear Regression,0.3519,0.09,0.02
3,IoU,Decision Tree,-0.3400,0.13,0.03
4,IoU,Random Forest,0.3423,0.09,0.02
5,IoU,MLP,0.0385,0.11,0.02
6,LRP,Constant Mean Predictor,-0.0001,0.10,0.02
7,LRP,Univariate Linear Regression (Confidence),0.4008,0.07,0.01
8,LRP,Linear Regression,0.4205,0.07,0.01
9,LRP,Decision Tree,-0.0781,0.10,0.02
